# Optional Project - Colab Part1 (No Drive)

This notebook runs Task1 only: preprocess, tokenizer, and HF publishing.


In [ ]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
# DATA_CONFIG = "configs/data.yaml"
DATA_CONFIG = "configs/data_vocab1024.yaml"
# DATA_CONFIG = "configs/data_vocab8192.yaml"


In [2]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


Cloning into '/content/optionalproject'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 81 (delta 27), reused 74 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 230.08 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/optionalproject
Already on 'run'
Your branch is up to date with 'origin/run'.
From https://github.com/Peng-y-x/optionalproject
 * branch            run        -> FETCH_HEAD
Already up to date.
run
7ebbd8e


In [3]:
# 2) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -U datasets huggingface_hub cairosvg tokenizers


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B] 

In [6]:
# 3) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
import os
print("has_hf_token:", bool(os.getenv("HF_TOKEN")))


has_hf_token: True


In [9]:
# 4) Render sanity check (must be True before preprocess)
from src.data.validate_svg import validate_render
svg = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24"><circle cx="12" cy="12" r="6"/></svg>'
ok, err = validate_render(svg)
print('render_check_ok:', ok)
print('render_check_err:', err)


render_check_ok: True
render_check_err: None


In [10]:
# 5) Preprocess data (and push clean dataset if hf_push.enabled=true)
%cd $REPO_DIR
# !python scripts/run_preprocess.py --config {DATA_CONFIG}
!python scripts/run_preprocess.py --config {DATA_CONFIG} --force

/content/optionalproject
[auth] Loaded HF token from Colab key.
[1/8] Checking processed cache + manifest...
  Force rebuild enabled. Regenerating outputs.
[2/8] Loading source datasets from Hugging Face (uses cache_dir for reuse)...
README.md: 1.76kB [00:00, 1.32MB/s]
data/train-00000-of-00001.parquet: 100% 137M/137M [00:08<00:00, 17.1MB/s]
data/test-00000-of-00001.parquet: 100% 4.59M/4.59M [00:00<00:00, 7.51MB/s]
data/val-00000-of-00001.parquet: 100% 11.1M/11.1M [00:00<00:00, 18.2MB/s]
Generating train split: 100% 80434/80434 [00:00<00:00, 88652.16 examples/s]
Generating test split: 100% 2682/2682 [00:00<00:00, 82178.77 examples/s]
Generating val split: 100% 6254/6254 [00:00<00:00, 88324.63 examples/s]
README.md: 1.76kB [00:00, 5.29MB/s]
data/train-00000-of-00001.parquet: 100% 12.7M/12.7M [00:03<00:00, 3.96MB/s]
data/test-00000-of-00001.parquet: 100% 1.05M/1.05M [00:00<00:00, 2.56MB/s]
data/val-00000-of-00001.parquet: 100% 687k/687k [00:00<00:00, 1.67MB/s]
Generating train split: 100

In [ ]:
# 5) or download preprocessed data from huggingface
from datasets import load_dataset
import json, os

repo_id = "Zala0429/svg-scaling-v1-clean"
out_dir = "data/processed/v1-clean-rawsplit"
os.makedirs(out_dir, exist_ok=True)

ds = load_dataset(repo_id)
for split, fname in [("train","train.jsonl"), ("validation","validation.jsonl"), ("test","test.jsonl")]:
    with open(f"{out_dir}/{fname}", "w", encoding="utf-8") as f:
        for row in ds[split]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [11]:
# 6) Train tokenizer + encode splits (enforces real train token target)
%cd $REPO_DIR
!python scripts/run_tokenizer.py --config {DATA_CONFIG}


/content/optionalproject
[1/4] Preparing tokenizer training text...
[2/4] Training BPE tokenizer...
[00:00:03] Tokenize words                 ██████████████████ 1509077  /  1509077[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:02] Count pairs                    ██████████████████ 1509077  /  1509077
[00:00:13] Compute merges                 ██████████████████ 4046     /     4046
  tokenizer: data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json
[3/4] Encoding train/validation/test splits...
[4/4] Writing tokenization stats...
Done.
Vocab size: 4096
Token totals:
  train: 100211019
  validation: 1034238
  test: 1025425


In [12]:
# 7) Push tokenizer artifacts to HF model repo
%cd $REPO_DIR
!python scripts/push_tokenizer_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
No files have been modified since last commit. Skipping to prevent empty commit.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:10913: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")
No files have been modified since last commit. Skipping to prevent empty commit.
Pushed tokenizer artifacts to: Zala0429/svg-scaling-tokenizer-v1
{"repo_id": "Zala0429/svg-scaling-tokenizer-v1", "repo_type": "model"}


In [13]:
# 8) Push tokenized dataset to HF dataset repo
%cd $REPO_DIR
!python scripts/push_tokenized_dataset_to_hf.py --config {DATA_CONFIG}


/content/optionalproject
[auth] Loaded HF token from Colab key.
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Uploading the dataset shards:   0% 0/1 [00:00<?, ? shards/s]
Creating parquet from Arrow format:   0% 0/5 [00:00<?, ?ba/s]
Creating parquet from Arrow format:  20% 1/5 [00:00<00:02,  1.43ba/s]
Creating parquet from Arrow format:  40% 2/5 [00:01<00:02,  1.45ba/s]
Creating parquet from Arrow format:  60% 3/5 [00:02<00:01,  1.49ba/s]
Creating parquet from Arrow format: 100% 5/5 [00:02<00:00,  1.83ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpl5dv18aj.parquet    :   0% 1.95M/407M [00:00<?, ?B/s]

Processing Files (0 / 1)      :   0% 1.95M/407M [00:01<06:17, 1.07MB/s, 1.22MB/s  ]
New Data Upload               :   3% 1.95M/67.0M [00:01<01:00, 1.07MB/s, 1.22MB/s  ]

  /tmp/tmpl5dv18aj.parquet    :   0% 1.

In [14]:
# 9) Inspect key outputs (auto-resolve tokenizer output_dir from DATA_CONFIG)
import json
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path(DATA_CONFIG).read_text(encoding='utf-8'))
root = Path(cfg['output']['dir'])
tokenizer_out_dir = Path(cfg['tokenization']['output_dir'])

stats = root / 'stats.json'
tok = tokenizer_out_dir / 'token_stats.json'
print('data_output_dir:', root)
print('tokenizer_output_dir:', tokenizer_out_dir)
print('stats exists:', stats.exists())
print('token_stats exists:', tok.exists())
if stats.exists():
    s = json.loads(stats.read_text(encoding='utf-8'))
    print('cleaned_records:', s.get('cleaned_records'))
    print('train_token_est_total:', s.get('train_token_est_total'))
if tok.exists():
    t = json.loads(tok.read_text(encoding='utf-8'))
    print('vocab_size:', t.get('vocab_size'))
    print('train_total_tokens:', t.get('splits', {}).get('train', {}).get('total_tokens'))


stats exists: True
token_stats exists: True
cleaned_records: 153020
train_token_est_total: 1892164
vocab_size: 4096
train_total_tokens: 100211019
